# 03 - Model Evaluation and Comparison

This notebook provides a comprehensive evaluation and comparison of all trained models in the AAI-521 Computer Vision Image Classification Project.

## Overview

This notebook evaluates and compares the performance of three trained models:
1. **Baseline CNN** - A custom convolutional neural network
2. **ResNet50** - Transfer learning with ResNet50 architecture
3. **EfficientNet-B2** - Transfer learning with EfficientNet-B2 architecture

We will:
- Load each trained model from saved checkpoints
- Evaluate performance on the test dataset
- Generate confusion matrices and classification reports
- Create comparative visualizations
- Identify the best performing model

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Load test data loader from data preparation
from src.data.loaders import get_data_loaders

# Get data loaders
train_loader, val_loader, test_loader = get_data_loaders(
    data_dir='../data',
    batch_size=32,
    num_workers=4
)

# Get class names
class_names = test_loader.dataset.classes if hasattr(test_loader.dataset, 'classes') else [f'Class_{i}' for i in range(10)]
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")
print(f"Test dataset size: {len(test_loader.dataset)}")

## Load Trained Models

We'll load each model architecture and its corresponding trained weights from the saved checkpoints.

### Load Baseline CNN Model

In [ ]:
# Load CNN model architecture and weights
from src.models.baseline_cnn import BaselineCNN

# Initialize CNN model
cnn_model = BaselineCNN(num_classes=num_classes).to(device)

# Load trained weights
cnn_checkpoint_path = '../models/cnn_best.pth'
if os.path.exists(cnn_checkpoint_path):
    cnn_model.load_state_dict(torch.load(cnn_checkpoint_path, map_location=device))
    cnn_model.eval()
    print(f"✓ CNN model loaded from {cnn_checkpoint_path}")
else:
    print(f"⚠ Warning: CNN checkpoint not found at {cnn_checkpoint_path}")

### Load ResNet50 Model

In [ ]:
# Load ResNet50 model architecture and weights
from torchvision import models

# Initialize ResNet50 model
resnet50_model = models.resnet50(pretrained=False)
resnet50_model.fc = nn.Linear(resnet50_model.fc.in_features, num_classes)
resnet50_model = resnet50_model.to(device)

# Load trained weights
resnet50_checkpoint_path = '../models/resnet50_best.pth'
if os.path.exists(resnet50_checkpoint_path):
    resnet50_model.load_state_dict(torch.load(resnet50_checkpoint_path, map_location=device))
    resnet50_model.eval()
    print(f"✓ ResNet50 model loaded from {resnet50_checkpoint_path}")
else:
    print(f"⚠ Warning: ResNet50 checkpoint not found at {resnet50_checkpoint_path}")

### Load EfficientNet-B2 Model

In [ ]:
# Load EfficientNet-B2 model architecture and weights
from torchvision import models

# Initialize EfficientNet-B2 model
efficientnet_model = models.efficientnet_b2(pretrained=False)
efficientnet_model.classifier[1] = nn.Linear(efficientnet_model.classifier[1].in_features, num_classes)
efficientnet_model = efficientnet_model.to(device)

# Load trained weights
efficientnet_checkpoint_path = '../models/efficientnet_b2_best.pth'
if os.path.exists(efficientnet_checkpoint_path):
    efficientnet_model.load_state_dict(torch.load(efficientnet_checkpoint_path, map_location=device))
    efficientnet_model.eval()
    print(f"✓ EfficientNet-B2 model loaded from {efficientnet_checkpoint_path}")
else:
    print(f"⚠ Warning: EfficientNet-B2 checkpoint not found at {efficientnet_checkpoint_path}")

## Evaluation Functions

Define helper functions for model evaluation and metrics calculation.

In [ ]:
def evaluate_model(model, data_loader, device):
    """
    Evaluate a model on the given data loader.
    
    Args:
        model: PyTorch model to evaluate
        data_loader: DataLoader containing test data
        device: Device to run evaluation on
    
    Returns:
        y_true: Ground truth labels
        y_pred: Predicted labels
        y_probs: Prediction probabilities
        accuracy: Overall accuracy
    """
    model.eval()
    y_true = []
    y_pred = []
    y_probs = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(data_loader, desc="Evaluating"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)
    
    accuracy = accuracy_score(y_true, y_pred)
    
    return y_true, y_pred, y_probs, accuracy


def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    """
    Plot a confusion matrix with annotations.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        class_names: List of class names
        title: Title for the plot
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title(title, fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


def calculate_metrics(y_true, y_pred):
    """
    Calculate comprehensive metrics for classification.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
    
    Returns:
        Dictionary containing accuracy, precision, recall, and F1 score
    """
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }
    return metrics

print("✓ Evaluation functions defined")

## Model Evaluations

Evaluate each model individually on the test dataset.

### Baseline CNN Evaluation

In [ ]:
# Evaluate CNN model
if os.path.exists(cnn_checkpoint_path):
    print("Evaluating Baseline CNN Model...")
    cnn_y_true, cnn_y_pred, cnn_y_probs, cnn_accuracy = evaluate_model(cnn_model, test_loader, device)
    
    print(f"\nBaseline CNN Test Accuracy: {cnn_accuracy:.4f} ({cnn_accuracy*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(cnn_y_true, cnn_y_pred, target_names=class_names, zero_division=0))
    
    # Plot confusion matrix
    plot_confusion_matrix(cnn_y_true, cnn_y_pred, class_names, title="Baseline CNN - Confusion Matrix")
    
    # Calculate detailed metrics
    cnn_metrics = calculate_metrics(cnn_y_true, cnn_y_pred)
    print("\nDetailed Metrics:")
    for metric, value in cnn_metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("CNN model not available for evaluation.")

### ResNet50 Evaluation

In [ ]:
# Evaluate ResNet50 model
if os.path.exists(resnet50_checkpoint_path):
    print("Evaluating ResNet50 Model...")
    resnet_y_true, resnet_y_pred, resnet_y_probs, resnet_accuracy = evaluate_model(resnet50_model, test_loader, device)
    
    print(f"\nResNet50 Test Accuracy: {resnet_accuracy:.4f} ({resnet_accuracy*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(resnet_y_true, resnet_y_pred, target_names=class_names, zero_division=0))
    
    # Plot confusion matrix
    plot_confusion_matrix(resnet_y_true, resnet_y_pred, class_names, title="ResNet50 - Confusion Matrix")
    
    # Calculate detailed metrics
    resnet_metrics = calculate_metrics(resnet_y_true, resnet_y_pred)
    print("\nDetailed Metrics:")
    for metric, value in resnet_metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("ResNet50 model not available for evaluation.")

### EfficientNet-B2 Evaluation

In [ ]:
# Evaluate EfficientNet-B2 model
if os.path.exists(efficientnet_checkpoint_path):
    print("Evaluating EfficientNet-B2 Model...")
    efficientnet_y_true, efficientnet_y_pred, efficientnet_y_probs, efficientnet_accuracy = evaluate_model(efficientnet_model, test_loader, device)
    
    print(f"\nEfficientNet-B2 Test Accuracy: {efficientnet_accuracy:.4f} ({efficientnet_accuracy*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(efficientnet_y_true, efficientnet_y_pred, target_names=class_names, zero_division=0))
    
    # Plot confusion matrix
    plot_confusion_matrix(efficientnet_y_true, efficientnet_y_pred, class_names, title="EfficientNet-B2 - Confusion Matrix")
    
    # Calculate detailed metrics
    efficientnet_metrics = calculate_metrics(efficientnet_y_true, efficientnet_y_pred)
    print("\nDetailed Metrics:")
    for metric, value in efficientnet_metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("EfficientNet-B2 model not available for evaluation.")

## Comprehensive Model Comparison

Compare all three models side-by-side using a summary table and visualizations.

### Comparison Table

In [ ]:
# Create comparison DataFrame
comparison_data = []

if os.path.exists(cnn_checkpoint_path):
    comparison_data.append({
        'Model': 'Baseline CNN',
        'Accuracy': cnn_metrics['Accuracy'],
        'Precision': cnn_metrics['Precision'],
        'Recall': cnn_metrics['Recall'],
        'F1-Score': cnn_metrics['F1-Score']
    })

if os.path.exists(resnet50_checkpoint_path):
    comparison_data.append({
        'Model': 'ResNet50',
        'Accuracy': resnet_metrics['Accuracy'],
        'Precision': resnet_metrics['Precision'],
        'Recall': resnet_metrics['Recall'],
        'F1-Score': resnet_metrics['F1-Score']
    })

if os.path.exists(efficientnet_checkpoint_path):
    comparison_data.append({
        'Model': 'EfficientNet-B2',
        'Accuracy': efficientnet_metrics['Accuracy'],
        'Precision': efficientnet_metrics['Precision'],
        'Recall': efficientnet_metrics['Recall'],
        'F1-Score': efficientnet_metrics['F1-Score']
    })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')

# Format as percentages
comparison_df_pct = comparison_df * 100

print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)
print(comparison_df_pct.to_string(float_format=lambda x: f'{x:.2f}%'))
print("=" * 80)

# Highlight best performing model
best_model = comparison_df_pct['Accuracy'].idxmax()
best_accuracy = comparison_df_pct['Accuracy'].max()
print(f"\n🏆 Best Model: {best_model} with {best_accuracy:.2f}% accuracy")

# Display styled DataFrame
display(comparison_df.style.format("{:.4f}").highlight_max(axis=0, props='background-color: lightgreen; font-weight: bold'))

### Visualization Comparisons

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

models_data = []
if os.path.exists(cnn_checkpoint_path):
    models_data.append(('Baseline CNN', cnn_y_true, cnn_y_pred))
if os.path.exists(resnet50_checkpoint_path):
    models_data.append(('ResNet50', resnet_y_true, resnet_y_pred))
if os.path.exists(efficientnet_checkpoint_path):
    models_data.append(('EfficientNet-B2', efficientnet_y_true, efficientnet_y_pred))

for idx, (model_name, y_true, y_pred) in enumerate(models_data):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('True Label', fontsize=10)
    axes[idx].set_xlabel('Predicted Label', fontsize=10)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Metrics comparison bar chart
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, metric in enumerate(metrics_list):
    ax = axes[idx // 2, idx % 2]
    values = comparison_df[metric].values * 100
    models = comparison_df.index.tolist()
    
    bars = ax.bar(models, values, color=colors[:len(models)], alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylabel(f'{metric} (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 105])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Overall metrics comparison - radar chart
from math import pi

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
N = len(categories)

# Calculate angles for each metric
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Plot each model
colors_radar = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for idx, (model_name, color) in enumerate(zip(comparison_df.index, colors_radar[:len(comparison_df)])):
    values = comparison_df.loc[model_name, categories].values.tolist()
    values += values[:1]  # Complete the circle
    
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], size=10)
ax.set_title('Model Performance Radar Chart', size=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Conclusion and Recommendations

### Summary of Results

Based on the comprehensive evaluation of all three models on the test dataset, we can draw the following conclusions:

In [ ]:
# Generate automated insights
print("=" * 80)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("=" * 80)

if len(comparison_df) > 0:
    # Best model
    best_model = comparison_df['Accuracy'].idxmax()
    best_acc = comparison_df.loc[best_model, 'Accuracy'] * 100
    
    print(f"\n1. BEST OVERALL MODEL: {best_model}")
    print(f"   - Achieved the highest accuracy: {best_acc:.2f}%")
    print(f"   - Precision: {comparison_df.loc[best_model, 'Precision']*100:.2f}%")
    print(f"   - Recall: {comparison_df.loc[best_model, 'Recall']*100:.2f}%")
    print(f"   - F1-Score: {comparison_df.loc[best_model, 'F1-Score']*100:.2f}%")
    
    # Performance ranking
    print("\n2. PERFORMANCE RANKING (by Accuracy):")
    sorted_models = comparison_df.sort_values('Accuracy', ascending=False)
    for rank, (model, row) in enumerate(sorted_models.iterrows(), 1):
        print(f"   {rank}. {model}: {row['Accuracy']*100:.2f}%")
    
    # Improvement analysis
    if len(comparison_df) >= 2:
        worst_model = comparison_df['Accuracy'].idxmin()
        improvement = (comparison_df.loc[best_model, 'Accuracy'] - 
                      comparison_df.loc[worst_model, 'Accuracy']) * 100
        print(f"\n3. PERFORMANCE GAP:")
        print(f"   - {best_model} outperforms {worst_model} by {improvement:.2f} percentage points")
    
    # Recommendations
    print("\n4. RECOMMENDATIONS:")
    print(f"   ✓ Deploy {best_model} for production use")
    print(f"   ✓ Consider ensemble methods combining top-performing models")
    print(f"   ✓ Analyze misclassified examples to identify improvement areas")
    print(f"   ✓ Experiment with hyperparameter tuning for {best_model}")
    
    # Model-specific insights
    print("\n5. MODEL-SPECIFIC INSIGHTS:")
    for model in comparison_df.index:
        f1 = comparison_df.loc[model, 'F1-Score'] * 100
        if f1 >= 90:
            print(f"   - {model}: Excellent performance across all metrics")
        elif f1 >= 80:
            print(f"   - {model}: Strong performance, suitable for deployment")
        elif f1 >= 70:
            print(f"   - {model}: Moderate performance, requires optimization")
        else:
            print(f"   - {model}: Needs significant improvement")

print("\n" + "=" * 80)

### Final Recommendations

#### Best Model Selection
The evaluation clearly identifies the best performing model based on multiple metrics including accuracy, precision, recall, and F1-score. This model should be prioritized for deployment in production environments.

#### Key Findings
1. **Transfer Learning Advantage**: Pre-trained models (ResNet50, EfficientNet-B2) typically outperform baseline CNN due to learned feature representations from large-scale datasets
2. **Balanced Performance**: Consider not just accuracy but also precision and recall, especially if class imbalance exists
3. **Computational Efficiency**: While EfficientNet-B2 is designed for efficiency, ResNet50 may offer better throughput depending on hardware

#### Next Steps
1. **Error Analysis**: Examine misclassified images to understand failure modes
2. **Ensemble Methods**: Combine predictions from multiple models for improved accuracy
3. **Hyperparameter Optimization**: Fine-tune learning rates, batch sizes, and augmentation strategies
4. **Data Augmentation**: Enhance training with more diverse augmentation techniques
5. **Cross-Validation**: Perform k-fold cross-validation for robust performance estimates
6. **Production Deployment**: Package the best model with proper versioning and monitoring

#### Deployment Considerations
- **Inference Speed**: Measure actual inference time on target hardware
- **Model Size**: Consider model compression techniques (pruning, quantization) if needed
- **Monitoring**: Implement performance monitoring to detect model drift over time
- **A/B Testing**: Compare model versions in production with careful experimentation

---

**Notebook Complete** ✓

This comprehensive evaluation provides all necessary information to make informed decisions about model selection and deployment for the image classification project.